# Split 2 — Trust-Weighted Aggregation Engine
## Blockchain-Based Dynamic Trust Modeling for Federated Fraud Detection

**What this notebook does:**
- Replaces FedAvg (Split 1) with trust-weighted adaptive aggregation
- Computes per-client anomaly scores using cosine similarity + euclidean distance + norm ratio
- Simulates adversarial attacks: label-flipping and gradient scaling
- Detects and neutralises malicious clients via dynamic trust weights
- Produces `trust_training_log.json` — consumed by Split 3 blockchain layer

**Trust scoring math:**
```
α_i = λ₁·(1 - cos_sim)/2  +  λ₂·norm(euc_dist)  +  λ₃·norm_penalty
τ_i(t+1) = γ·τ_i(t) + (1-γ)·(1 - α_i)
w_i = τ_i / Σ τ_j
```

---
### Pipeline
```
Split 1 data partitions + local models
         │
         ▼
 AttackSimulator ──► poisoned client data/gradients (malicious clients)
         │
         ▼
 TrustScorer ──► cosine_sim, euc_dist, norm_ratio → anomaly_score α_i
         │
         ▼
 TrustWeightedFedAvg ──► w_i = τ_i / Σ τ_j  →  global model
         │
         ▼
 trust_training_log.json  →  Split 3 Blockchain Governance
```

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

install('flwr[simulation]')
install('torch')
install('scikit-learn')
install('imbalanced-learn')
install('matplotlib')
install('pandas')

print('✅ All dependencies installed')

In [ ]:
# ── Cell 2: Mount Drive and set paths ────────────────────────────────────────
import os, sys

USE_DRIVE = False  # Set True if your files are in Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/fraud_detection'
else:
    PROJECT_ROOT = '/content'

SPLIT1_DIR = os.path.join(PROJECT_ROOT, 'split1_federated_core')
SPLIT2_DIR = os.path.join(PROJECT_ROOT, 'split2_trust_aggregation')
LOG_DIR    = os.path.join(PROJECT_ROOT, 'logs_split2')
DATA_PATH  = os.path.join(PROJECT_ROOT, 'data', 'creditcard.csv')

os.makedirs(LOG_DIR,    exist_ok=True)
os.makedirs(SPLIT2_DIR, exist_ok=True)

# Add both splits to path (Split 2 imports from Split 1)
sys.path.insert(0, SPLIT2_DIR)
sys.path.insert(0, SPLIT1_DIR)

print(f'Split 1 dir : {SPLIT1_DIR}  (exists={os.path.exists(SPLIT1_DIR)})')
print(f'Split 2 dir : {SPLIT2_DIR}  (exists={os.path.exists(SPLIT2_DIR)})')
print(f'Log dir     : {LOG_DIR}')
print(f'Dataset     : {DATA_PATH}  (exists={os.path.exists(DATA_PATH)})')

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────

# ── Training config ───────────────────────────────────────────────────────────
NUM_CLIENTS   = 5
NUM_ROUNDS    = 10      # Increase to 25 for full convergence
ALPHA_DIR     = 0.5     # Dirichlet alpha for data partitioning
USE_SYNTHETIC = not os.path.exists(DATA_PATH)  # Auto-fallback to synthetic

# ── Attack config ─────────────────────────────────────────────────────────────
ATTACK_TYPE       = 'label_flip'   # 'none' | 'label_flip' | 'gradient_scale' | 'combined'
MALICIOUS_CLIENTS = [1]            # Which clients attack (empty list = clean run)
SCALE_FACTOR      = 5.0            # For gradient_scale attack

# ── Trust scoring hyperparameters ─────────────────────────────────────────────
LAMBDA_COSINE   = 0.5   # λ₁ — weight of cosine direction penalty
LAMBDA_DISTANCE = 0.3   # λ₂ — weight of euclidean distance penalty
LAMBDA_NORM     = 0.2   # λ₃ — weight of norm ratio penalty
GAMMA           = 0.9   # γ  — trust decay factor (higher = slower to trust/distrust)
ANOMALY_THRESHOLD = 0.6 # α threshold → flag as malicious

# ── Comparison ────────────────────────────────────────────────────────────────
RUN_BASELINE_COMPARISON = True  # Also run standard FedAvg for comparison

print('Configuration:')
print(f'  Clients:        {NUM_CLIENTS}')
print(f'  Rounds:         {NUM_ROUNDS}')
print(f'  Dataset:        {"Synthetic" if USE_SYNTHETIC else DATA_PATH}')
print(f'  Attack:         {ATTACK_TYPE}')
print(f'  Malicious:      {MALICIOUS_CLIENTS}')
print(f'  Trust λ:        cos={LAMBDA_COSINE} dist={LAMBDA_DISTANCE} norm={LAMBDA_NORM}')
print(f'  γ (decay):      {GAMMA}')
print(f'  Threshold α:    {ANOMALY_THRESHOLD}')

In [ ]:
# ── Cell 4: Load / generate dataset and partition ─────────────────────────────
import numpy as np

if USE_SYNTHETIC:
    print('Generating synthetic fraud dataset...')
    rng = np.random.default_rng(42)
    N, F, FRAUD_RATE = 20_000, 30, 0.03
    X = rng.standard_normal((N, F)).astype(np.float32)
    y = rng.choice([0, 1], size=N, p=[1-FRAUD_RATE, FRAUD_RATE]).astype(int)
    X[y == 1] += rng.standard_normal((int(y.sum()), F)) * 0.5 + 1.5
    print(f'  Samples: {N:,} | Features: {F} | Fraud rate: {y.mean()*100:.2f}%')
else:
    print(f'Loading dataset: {DATA_PATH}')
    from data_partition import load_creditcard_dataset
    X, y = load_creditcard_dataset(DATA_PATH)
    print(f'  Loaded: {len(X):,} samples | Fraud rate: {y.mean()*100:.2f}%')

# Partition data across clients using Dirichlet distribution
from data_partition import dirichlet_partition
partitions = dirichlet_partition(X, y, num_clients=NUM_CLIENTS, alpha=ALPHA_DIR)

print(f'\nPartition sizes:')
for cid, p in enumerate(partitions):
    tr_fraud = int(p['y_train'].sum())
    te_fraud = int(p['y_test'].sum())
    print(f'  Client {cid}: train={len(p["X_train"]):,} ({tr_fraud} fraud) | '
          f'test={len(p["X_test"]):,} ({te_fraud} fraud)')

In [ ]:
# ── Cell 5: Setup AttackSimulator ─────────────────────────────────────────────
from attack_simulator import AttackSimulator

attacker = AttackSimulator(
    attack_type=ATTACK_TYPE,
    malicious_clients=MALICIOUS_CLIENTS,
    scale_factor=SCALE_FACTOR,
    attack_start_round=1,
)

print(f'\nAttack summary: {attacker.get_attack_summary()}')

In [ ]:
# ── Cell 6: Demo trust scoring math (before full training) ───────────────────
from trust_scoring import TrustScorer

demo_scorer = TrustScorer(
    num_clients=NUM_CLIENTS,
    lambda_cosine=LAMBDA_COSINE,
    lambda_distance=LAMBDA_DISTANCE,
    lambda_norm=LAMBDA_NORM,
    gamma=GAMMA,
    anomaly_threshold=ANOMALY_THRESHOLD,
)

print('Trust scoring demo with synthetic gradient vectors:')
print('─'*60)

rng_demo = np.random.default_rng(0)
dim = 100  # gradient dimension
global_g = rng_demo.standard_normal(dim)

test_cases = [
    ('Benign (aligned)',         global_g + rng_demo.normal(0, 0.1, dim)),
    ('Benign (slightly noisy)',  global_g + rng_demo.normal(0, 0.5, dim)),
    ('Label-flip (opposite)',    -global_g + rng_demo.normal(0, 0.1, dim)),
    ('Gradient scale (5x)',      global_g * 5.0),
    ('Combined attack',          -global_g * 4.0 + rng_demo.normal(0, 0.2, dim)),
]

print(f'  {"Case":<30} {"cos_sim":>8} {"euc_dist":>10} {"norm_ratio":>12} {"α (anomaly)":>12}')
print('  ' + '-'*72)
for name, g in test_cases:
    cs = demo_scorer.cosine_similarity(g, global_g)
    ed = demo_scorer.euclidean_distance(g, global_g)
    nr = demo_scorer.norm_ratio(g, global_g)
    alpha = demo_scorer.anomaly_score(cs, ed, nr, euc_max=max(ed, 1.0))
    flag = '🚨' if alpha > ANOMALY_THRESHOLD else '✅'
    print(f'  {flag} {name:<28} {cs:>8.4f} {ed:>10.4f} {nr:>12.4f} {alpha:>12.4f}')

In [ ]:
# ── Cell 7: Define attacked client factory ────────────────────────────────────
import flwr as fl
from flower_client import BankFederatedClient
from data_partition import apply_smote

class AttackedBankClient(BankFederatedClient):
    """BankFederatedClient with data poisoning injected before training."""
    def __init__(self, client_id, X_train, y_train, X_test, y_test, model_type, use_smote):
        X_p, y_p = attacker.poison_data(client_id, X_train, y_train)
        super().__init__(client_id, X_p, y_p, X_test, y_test, model_type, use_smote)
        self._attacker = attacker

def make_client_fn_attacked(cid_str):
    cid = int(cid_str)
    p   = partitions[cid]
    return AttackedBankClient(
        client_id=cid,
        X_train=p['X_train'], y_train=p['y_train'],
        X_test=p['X_test'],   y_test=p['y_test'],
        model_type='dnn',
        use_smote=True,
    )

print('✅ Attacked client factory ready')

In [ ]:
# ── Cell 8: (Optional) Run FedAvg baseline for comparison ────────────────────
import os, json
from fedavg_strategy import get_fedavg_strategy

if RUN_BASELINE_COMPARISON:
    print('Running STANDARD FedAvg (baseline — no trust scoring)...')
    baseline_log_dir = os.path.join(LOG_DIR, 'baseline_fedavg')
    os.makedirs(baseline_log_dir, exist_ok=True)

    baseline_strategy = get_fedavg_strategy(
        num_clients=NUM_CLIENTS,
        log_dir=baseline_log_dir,
    )

    fl.simulation.start_simulation(
        client_fn=make_client_fn_attacked,
        num_clients=NUM_CLIENTS,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=baseline_strategy,
        client_resources={'num_cpus': 1, 'num_gpus': 0.0},
    )
    print('\n✅ Baseline FedAvg complete')
else:
    print('Skipping baseline (RUN_BASELINE_COMPARISON=False)')

In [ ]:
# ── Cell 9: Run Trust-Weighted Federated Learning ─────────────────────────────
from trust_weighted_strategy import TrustWeightedFedAvg

print('='*65)
print('  Running TRUST-WEIGHTED FedAvg')
print(f'  Attack: {ATTACK_TYPE} | Malicious: {MALICIOUS_CLIENTS}')
print('='*65)

trust_strategy = TrustWeightedFedAvg(
    num_clients=NUM_CLIENTS,
    lambda_cosine=LAMBDA_COSINE,
    lambda_distance=LAMBDA_DISTANCE,
    lambda_norm=LAMBDA_NORM,
    gamma=GAMMA,
    anomaly_threshold=ANOMALY_THRESHOLD,
    log_dir=LOG_DIR,
    attack_simulator=attacker,
)

fl.simulation.start_simulation(
    client_fn=make_client_fn_attacked,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=trust_strategy,
    client_resources={'num_cpus': 1, 'num_gpus': 0.0},
)

trust_strategy.print_summary()

In [ ]:
# ── Cell 10: Load and inspect trust_training_log.json ─────────────────────────

TRUST_LOG_PATH = os.path.join(LOG_DIR, 'trust_training_log.json')
with open(TRUST_LOG_PATH) as f:
    trust_log = json.load(f)

print(f'Trust log loaded: {len(trust_log)} rounds')
print(f'\nSample entry (Round 1):')
sample = {k: v for k, v in trust_log[0].items()
          if k not in ('cos_similarities', 'euc_distances')}
print(json.dumps(sample, indent=2))

In [ ]:
# ── Cell 11: Per-round metrics summary table ──────────────────────────────────
import pandas as pd

rows = []
for entry in trust_log:
    rnd = entry['round']
    rows.append({
        'Round':     rnd,
        'Global F1': round(entry.get('global_f1', 0), 4),
        'Global AUC': round(entry.get('global_auc', 0), 4),
        'Trusted':   str(entry.get('trusted_clients', [])),
        'Flagged':   str(entry.get('flagged_clients', [])) or 'none',
        'Max α':     round(max(entry.get('anomaly_scores', {'0': 0}).values()), 4),
        'Min τ':     round(min(entry.get('trust_scores',  {'0': 1}).values()), 4),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

best_f1    = max(entry['global_f1']  for entry in trust_log)
best_round = next(e['round'] for e in trust_log if e['global_f1'] == best_f1)
print(f'\nBest Global F1: {best_f1:.4f} @ Round {best_round}')

In [ ]:
# ── Cell 12: Per-client metrics final round ───────────────────────────────────

final_entry = trust_log[-1]
print(f'Per-client metrics — Final Round {final_entry["round"]}:')
print('─'*70)
print(f'  {"Client":>8} | {"Trust τ":>8} | {"Anomaly α":>10} | {"cos_sim":>8} | {"Status":>12}')
print('  ' + '-'*60)

for cid_str in sorted(final_entry.get('trust_scores', {}).keys(), key=int):
    cid = int(cid_str)
    tau   = final_entry['trust_scores'].get(cid_str, 1.0)
    alpha = final_entry['anomaly_scores'].get(cid_str, 0.0)
    cos   = final_entry.get('cos_similarities', {}).get(cid_str, 1.0)
    flag  = cid in [int(c) for c in final_entry.get('flagged_clients', [])]
    status = '🚨 MALICIOUS' if flag else '✅ trusted'
    print(f'  Client {cid:02d} | {tau:>8.4f} | {alpha:>10.4f} | {cos:>8.4f} | {status}')

In [ ]:
# ── Cell 13: Visualisations ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

rounds      = [e['round']      for e in trust_log]
f1_vals     = [e['global_f1']  for e in trust_log]
auc_vals    = [e['global_auc'] for e in trust_log]
client_ids  = sorted(set(int(k) for e in trust_log for k in e.get('trust_scores', {}).keys()))
colors      = plt.cm.tab10(np.linspace(0, 1, len(client_ids)))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(
    f'Split 2: Trust-Weighted Aggregation\n'
    f'Attack: {ATTACK_TYPE} | Malicious clients: {MALICIOUS_CLIENTS if MALICIOUS_CLIENTS else "none"}',
    fontsize=12, fontweight='bold'
)

# Plot 1: Global F1
ax = axes[0, 0]
ax.plot(rounds, f1_vals, 'o-', color='#2196F3', lw=2, ms=5, label='Trust-Weighted')
if RUN_BASELINE_COMPARISON:
    baseline_path = os.path.join(LOG_DIR, 'baseline_fedavg', 'training_log.json')
    if os.path.exists(baseline_path):
        with open(baseline_path) as f: b_log = json.load(f)
        b_r  = [e['round'] for e in b_log]
        b_f1 = [e.get('global_f1', 0) for e in b_log]
        ax.plot(b_r, b_f1, 's--', color='#FF5252', lw=2, ms=5, label='Standard FedAvg')
ax.set_title('Global F1 Score'); ax.set_xlabel('Round'); ax.set_ylabel('F1')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(True, alpha=0.3)

# Plot 2: Global AUC
ax = axes[0, 1]
ax.plot(rounds, auc_vals, 's-', color='#4CAF50', lw=2, ms=5)
ax.set_title('Global AUC-ROC'); ax.set_xlabel('Round'); ax.set_ylabel('AUC')
ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)

# Plot 3: Per-client trust scores (τ)
ax = axes[1, 0]
for cid, color in zip(client_ids, colors):
    tau_vals = [e['trust_scores'].get(str(cid), 1.0) for e in trust_log]
    style = '--' if cid in MALICIOUS_CLIENTS else '-'
    label = f'Client {cid} 🚨' if cid in MALICIOUS_CLIENTS else f'Client {cid}'
    ax.plot(rounds, tau_vals, style, color=color, lw=2, ms=4, marker='o', label=label)
ax.axhline(y=0.4, color='red', ls=':', alpha=0.7, label='Low trust boundary')
ax.set_title('Per-Client Trust Score (τ)'); ax.set_xlabel('Round'); ax.set_ylabel('τ')
ax.set_ylim(-0.05, 1.1); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# Plot 4: Per-client anomaly scores (α)
ax = axes[1, 1]
for cid, color in zip(client_ids, colors):
    alpha_vals = [e['anomaly_scores'].get(str(cid), 0.0) for e in trust_log]
    style = '--' if cid in MALICIOUS_CLIENTS else '-'
    label = f'Client {cid} 🚨' if cid in MALICIOUS_CLIENTS else f'Client {cid}'
    ax.plot(rounds, alpha_vals, style, color=color, lw=2, ms=4, marker='o', label=label)
ax.axhline(y=ANOMALY_THRESHOLD, color='red', ls=':', alpha=0.7, label=f'Threshold α={ANOMALY_THRESHOLD}')
ax.set_title('Per-Client Anomaly Score (α)'); ax.set_xlabel('Round'); ax.set_ylabel('α')
ax.set_ylim(-0.05, 1.1); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(LOG_DIR, 'trust_training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Plots saved: {plot_path}')

In [ ]:
# ── Cell 14: Trust weight heatmap ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Trust Weight and Anomaly Score Heatmaps', fontsize=12, fontweight='bold')

trust_matrix = np.array([
    [e['trust_weights'].get(str(cid), 0.0) for cid in client_ids]
    for e in trust_log
])
anomaly_matrix = np.array([
    [e['anomaly_scores'].get(str(cid), 0.0) for cid in client_ids]
    for e in trust_log
])

for ax, matrix, title, cmap in zip(
    axes,
    [trust_matrix, anomaly_matrix],
    ['Trust Weight w_i per Round', 'Anomaly Score α_i per Round'],
    ['YlGn', 'YlOrRd'],
):
    im = ax.imshow(matrix.T, aspect='auto', cmap=cmap, vmin=0, vmax=1)
    ax.set_yticks(range(len(client_ids)))
    ax.set_yticklabels(
        [f'Client {c} 🚨' if c in MALICIOUS_CLIENTS else f'Client {c}' for c in client_ids]
    )
    ax.set_xticks(range(len(rounds)))
    ax.set_xticklabels([str(r) for r in rounds], fontsize=8)
    ax.set_title(title); ax.set_xlabel('Round')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
heatmap_path = os.path.join(LOG_DIR, 'trust_heatmaps.png')
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Heatmap saved: {heatmap_path}')

In [ ]:
# ── Cell 15: Validate trust_training_log.json schema (Split 3 readiness) ─────

REQUIRED_KEYS = [
    'round', 'timestamp', 'model_hash',
    'trusted_clients', 'flagged_clients',
    'trust_weights', 'anomaly_scores',
    'cos_similarities', 'euc_distances',
    'trust_scores', 'global_f1', 'global_auc',
]

print('Validating trust_training_log.json for Split 3 compatibility...')
print('─'*60)
all_ok = True
for entry in trust_log:
    rnd = entry.get('round', '?')
    missing = [k for k in REQUIRED_KEYS if k not in entry]
    if missing:
        print(f'  ❌ Round {rnd}: missing keys: {missing}')
        all_ok = False
    # Validate model_hash is a 64-char hex string
    mh = entry.get('model_hash', '')
    if not (isinstance(mh, str) and len(mh) == 64):
        print(f'  ❌ Round {rnd}: model_hash invalid: {mh[:20]}...')
        all_ok = False

if all_ok:
    print(f'  ✅ All {len(trust_log)} rounds pass schema validation')
    print(f'  ✅ trust_training_log.json is ready for Split 3')
else:
    print(f'  ⚠️  Fix the above issues before running Split 3')

In [ ]:
# ── Cell 16: Performance vs expected benchmarks ───────────────────────────────

BENCHMARKS = {
    'Kaggle Credit Card Fraud': {'f1': (0.75, 0.92), 'auc': (0.95, 0.99)},
    'IEEE-CIS':                 {'f1': (0.60, 0.80), 'auc': (0.88, 0.95)},
    'Synthetic':                {'f1': (0.45, 0.80), 'auc': (0.70, 0.92)},
}

dataset_key = 'Synthetic' if USE_SYNTHETIC else 'Kaggle Credit Card Fraud'
bench = BENCHMARKS[dataset_key]

f1_lo,  f1_hi  = bench['f1']
auc_lo, auc_hi = bench['auc']

f1_pass  = f1_lo  <= best_f1  <= f1_hi
auc_pass = auc_lo <= max(auc_vals) <= auc_hi

print(f'Performance vs expected benchmarks ({dataset_key}):')
print('─'*55)
print(f'  Best Global F1:  {best_f1:.4f}   expected [{f1_lo}–{f1_hi}]   {"✅" if f1_pass else "⚠️"}')
print(f'  Best Global AUC: {max(auc_vals):.4f}   expected [{auc_lo}–{auc_hi}]   {"✅" if auc_pass else "⚠️"}')

if not f1_pass and best_f1 > f1_hi:
    print('  ℹ️  F1 above upper bound — normal for clean synthetic data')
elif not f1_pass:
    print('  ⚠️  F1 below lower bound — may need more rounds or tuning')

In [ ]:
# ── Cell 17: Attack effectiveness analysis ────────────────────────────────────

if MALICIOUS_CLIENTS and ATTACK_TYPE != 'none':
    print(f'Attack effectiveness analysis — {ATTACK_TYPE}:')
    print('─'*60)

    flagged_counts = {cid: 0 for cid in range(NUM_CLIENTS)}
    for entry in trust_log:
        for cid in entry.get('flagged_clients', []):
            flagged_counts[int(cid)] += 1

    print(f'  Client flagged counts over {NUM_ROUNDS} rounds:')
    for cid, count in sorted(flagged_counts.items()):
        expected = '(malicious)' if cid in MALICIOUS_CLIENTS else '(benign)'
        detected = count > 0
        mark = '✅ DETECTED' if (detected and cid in MALICIOUS_CLIENTS) else \
               ('✅ clean'   if not detected and cid not in MALICIOUS_CLIENTS) else \
               '⚠️  FALSE POSITIVE'
        print(f'    Client {cid} {expected:>12} : flagged {count:>2}/{NUM_ROUNDS} rounds  {mark}')

    # Final trust score of malicious clients
    print(f'\n  Final trust weights of malicious clients:')
    final_tw = trust_log[-1].get('trust_weights', {})
    for cid in MALICIOUS_CLIENTS:
        w = final_tw.get(str(cid), 1.0/NUM_CLIENTS)
        print(f'    Client {cid}: w={w:.6f}  {"(near zero → neutralised ✅)" if w < 0.01 else "(still active ⚠️)"}')
else:
    print('No attack configured — skipping effectiveness analysis.')

In [ ]:
# ── Cell 18: Final summary ────────────────────────────────────────────────────

SEP = '='*65
print(SEP)
print('  SPLIT 2 — TRUST-WEIGHTED AGGREGATION COMPLETE')
print(SEP)
print(f'  Rounds:              {NUM_ROUNDS}')
print(f'  Best Global F1:      {best_f1:.4f} (Round {best_round})')
print(f'  Best Global AUC:     {max(auc_vals):.4f}')
print(f'  Attack type:         {ATTACK_TYPE}')
print(f'  Malicious clients:   {MALICIOUS_CLIENTS}')
print(f'  Trust log:           {TRUST_LOG_PATH}')
print(f'''
  Outputs:
    trust_training_log.json  →  Split 3 blockchain input
    trust_training_curves.png
    trust_heatmaps.png

  Next:
    Run split3_colab.ipynb to register model hashes on
    the simulated Hyperledger Fabric blockchain and build
    the SHA-256 hash chain with tamper detection.
''')
print(SEP)